# Day 033 — Exercise 5: BatchProcessor

**What you'll build:** The `BatchProcessor` class — `process(items, prompt_fn)` (async, for notebooks) and `run(items, prompt_fn)` (sync, for scripts). `prompt_fn` converts each item to a prompt string; `BatchProcessor` handles the concurrent LLM calls.

**Why it matters:** Separating *what to ask* (`prompt_fn`) from *how to batch* (`BatchProcessor`) makes the class reusable across any batch task — just swap the `prompt_fn`.

## Provided: All Helper Functions

In [ ]:
import asyncio
import ollama

async def async_chat(prompt: str, model: str = "llama3.2") -> str:
    client = ollama.AsyncClient()
    response = await client.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"]


import asyncio

async def gather_results(coros: list) -> list:
    return list(await asyncio.gather(*coros))


async def throttled_gather(coros: list, max_concurrent: int) -> list:
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(coro):
        async with sem:
            return await coro
    return list(await asyncio.gather(*[_run(c) for c in coros]))


async def process_batch(
    items: list, async_fn, max_concurrent: int = 3
) -> list[dict]:
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(item):
        async with sem:
            try:
                result = await async_fn(item)
                return {"item": item, "status": "ok",
                        "result": result, "error": None}
            except Exception as e:
                return {"item": item, "status": "error",
                        "result": None, "error": str(e)}
    return list(await asyncio.gather(*[_run(i) for i in items]))

## Your Implementation

In [ ]:
class BatchProcessor:
    """
    Async batch processor. Applies prompt_fn to each item, calls async_chat,
    and returns error-envelope dicts via process_batch.
    """

    def __init__(self, max_concurrent: int = 3, model: str = 'llama3.2'):
        # TODO: self.max_concurrent = max_concurrent
        # TODO: self.model = model
        pass

    async def process(self, items: list, prompt_fn) -> list[dict]:
        # TODO: async def _call(item):
        #     return await async_chat(prompt_fn(item), self.model)
        # TODO: return await process_batch(items, _call, self.max_concurrent)
        pass

    def run(self, items: list, prompt_fn) -> list[dict]:
        # TODO: return asyncio.run(self.process(items, prompt_fn))
        # Note: do NOT call run() from inside Jupyter — use await process() instead
        pass

## Check Your Work

In [ ]:
import asyncio

async def _run_checks():
    total = 5
    passed = 0

    # Check 1: class defined with all 3 methods
    try:
        assert 'BatchProcessor' in globals()
        for m in ('process', 'run'):
            assert hasattr(BatchProcessor, m), f'missing method: {m}'
        passed += 1; print('\u2705 Check 1: BatchProcessor with process and run methods')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    # Check 2: __init__ stores max_concurrent and model
    try:
        bp = BatchProcessor(max_concurrent=5, model='llama3.2')
        assert bp.max_concurrent == 5, \
            f'max_concurrent: expected 5, got {bp.max_concurrent}'
        assert bp.model == 'llama3.2', \
            f'model: expected llama3.2, got {bp.model!r}'
        bp_def = BatchProcessor()
        assert bp_def.max_concurrent == 3, \
            f'default max_concurrent: expected 3, got {bp_def.max_concurrent}'
        passed += 1; print('\u2705 Check 2: __init__ stores max_concurrent and model with defaults')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: process is an async method
    try:
        bp = BatchProcessor()
        assert asyncio.iscoroutinefunction(bp.process), \
            'process must be async def'
        coro = bp.process([], lambda x: x)
        assert asyncio.iscoroutine(coro), \
            f'process() should return a coroutine, got {type(coro)}'
        await coro  # run empty batch
        passed += 1; print('\u2705 Check 3: process is an async method')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: await process returns list[dict] with correct structure
    try:
        bp = BatchProcessor(max_concurrent=2)
        items = ['cat', 'dog']
        results = await bp.process(
            items, lambda x: f"What animal is a {x}? One word only."
        )
        assert isinstance(results, list), \
            f'expected list, got {type(results).__name__}'
        assert len(results) == 2, \
            f'expected 2 results, got {len(results)}'
        for r in results:
            assert 'item'   in r, f'missing item key: {r}'
            assert 'status' in r, f'missing status key: {r}'
            assert 'result' in r, f'missing result key: {r}'
            assert 'error'  in r, f'missing error key: {r}'
        passed += 1; print('\u2705 Check 4: await process returns list[dict] with correct structure')
    except Exception as e:
        print(f'\u274c Check 4: {e}')
        results = []

    # Check 5: successful results have status='ok' and string result
    try:
        ok_results = [r for r in results if r['status'] == 'ok']
        assert len(ok_results) > 0, \
            'no ok results — did the LLM calls succeed?'
        for r in ok_results:
            assert isinstance(r['result'], str), \
                f"ok result should be str: {r['result']!r}"
            assert r['error'] is None, \
                f"ok result error should be None: {r}"
        passed += 1; print(f'\u2705 Check 5: {len(ok_results)}/2 ok results with string content')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


await _run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
class BatchProcessor:
    def __init__(self, max_concurrent: int = 3, model: str = "llama3.2"):
        self.max_concurrent = max_concurrent
        self.model          = model

    async def process(self, items: list, prompt_fn) -> list[dict]:
        async def _call(item):
            return await async_chat(prompt_fn(item), self.model)
        return await process_batch(items, _call, self.max_concurrent)

    def run(self, items: list, prompt_fn) -> list[dict]:
        return asyncio.run(self.process(items, prompt_fn))
```

</details>